# SoilGrids

No account needed. pH, SOC, nitrogen, clay, sand, CEC, bulk density - read straight from ISRIC's remote VRT via rasterio and cropped to the Banten bbox (not a global download).

Needs the `rasterio` package (see requirements.txt).

In [ ]:
# make sure we're running from scripts/download so `import common` resolves
import os, sys

NOTEBOOK_DIR = os.path.abspath(".")
if os.path.basename(NOTEBOOK_DIR) != "download":
    NOTEBOOK_DIR = os.path.join(NOTEBOOK_DIR, "scripts", "download")
os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
print("cwd:", os.getcwd())

In [ ]:
from common import BANTEN_BBOX, RAW_DIR, ensure_dir, get_logger

log = get_logger("soilgrids")

SOILGRIDS_BASE = "https://files.isric.org/soilgrids/latest/data"
OUT_DIR = RAW_DIR / "soilgrids"

# (ISRIC property name, depth)
PROPERTIES = [
    ("phh2o", "0-5cm"),
    ("soc", "0-5cm"),
    ("nitrogen", "0-5cm"),
    ("clay", "0-5cm"),
    ("sand", "0-5cm"),
    ("cec", "0-5cm"),
    ("bdod", "0-5cm"),
]
STAT = "mean"


def download_property(prop: str, depth: str) -> None:
    import rasterio
    from rasterio.vrt import WarpedVRT
    from rasterio.windows import from_bounds

    url = f"/vsicurl/{SOILGRIDS_BASE}/{prop}/{prop}_{depth}_{STAT}.vrt"
    dest = OUT_DIR / f"{prop}_{depth}_{STAT}.tif"
    if dest.exists():
        log.info("Already have it, skipping: %s", dest)
        return

    log.info("Reading %s (remote, can take a bit)...", url)
    with rasterio.open(url) as src, WarpedVRT(src, crs="EPSG:4326") as vrt:
        window = from_bounds(
            BANTEN_BBOX["west"],
            BANTEN_BBOX["south"],
            BANTEN_BBOX["east"],
            BANTEN_BBOX["north"],
            transform=vrt.transform,
        )
        data = vrt.read(1, window=window)
        transform = vrt.window_transform(window)
        profile = vrt.profile.copy()
        profile.update(
            {
                "height": data.shape[0],
                "width": data.shape[1],
                "transform": transform,
                "driver": "GTiff",
                "compress": "deflate",
            }
        )
        with rasterio.open(dest, "w", **profile) as dst:
            dst.write(data, 1)
    log.info("Done: %s", dest)


def main() -> None:
    try:
        import rasterio  # noqa: F401
    except ImportError:
        log.error("The 'rasterio' package isn't installed. Run: pip install rasterio")
        return

    ensure_dir(OUT_DIR)
    for prop, depth in PROPERTIES:
        try:
            download_property(prop, depth)
        except Exception as e:  # noqa: BLE001
            log.warning("Failed to fetch %s (%s): %s", prop, depth, e)

In [ ]:
main()